In [ ]:
"""
M1 — U-NET CARDIAQUE ROBUSTE (version corrigée)
================================================
Corrections par rapport à la version initiale :

  C1. Transformations géométriques APPARIÉES image/masque
      (auparavant : flip et rotation appliqués à l'image seule)
  C2. Flip horizontal désactivé par défaut — anatomiquement invalide
      sur une radiographie thoracique (le cœur passerait à droite)
  C3. Masques VinDr par vote majoritaire des annotateurs
      (auparavant : union des trois boîtes → vérité terrain gonflée)
  C4. Split sans fuite : le test JSRT n'entre plus dans la validation
  C5. Checkpoints best / final réellement distincts
  C6. Dice et IoU calculés par image puis moyennés

Kaggle Notebook, GPU T4.
"""
import os, json, random
import numpy as np
import pandas as pd
import cv2
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================
BASE_JSRT        = '/kaggle/input/datasets/abdallahouahmane/segmentation/segmentation02/segmentation'
JSRT_TRAIN_IMGS  = f'{BASE_JSRT}/org_train'
JSRT_TRAIN_MASKS = f'{BASE_JSRT}/label_train'
JSRT_TEST_IMGS   = f'{BASE_JSRT}/org_test'
JSRT_TEST_MASKS  = f'{BASE_JSRT}/label_test'

VINDR_BASE  = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection'
VINDR_CSV   = f'{VINDR_BASE}/train.csv'
VINDR_DIR   = f'{VINDR_BASE}/train'

HEART_VALUE    = 85
TARGET_CLASSES = ['Cardiomegaly']

IMG_SIZE   = 512
EPOCHS     = 30
LR         = 1e-4
BATCH      = 8
SEED       = 42
PATIENCE   = 3          # cohérent avec §5.1.2 du rapport
MIN_VOTES  = 2          # C3 : consensus 2 annotateurs sur 3
USE_HFLIP  = False      # C2 : voir justification en tête de fichier
JSRT_VAL_N = 20         # C4 : images JSRT réservées à la validation

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")


# ============================================================
# C1 — AUGMENTATION APPARIÉE
# ============================================================
def augment_pair(img, mask, rotation, translate, brightness, contrast,
                 use_hflip=USE_HFLIP):
    """
    Applique EXACTEMENT la même transformation géométrique à l'image
    et au masque, en tirant les paramètres une seule fois.
    Les transformations photométriques ne touchent que l'image.
    """
    if use_hflip and random.random() < 0.5:
        img, mask = TF.hflip(img), TF.hflip(mask)

    angle = random.uniform(-rotation, rotation)
    w, h  = img.size
    tx    = int(random.uniform(-translate * w, translate * w))
    ty    = int(random.uniform(-translate * h, translate * h))

    img  = TF.affine(img,  angle=angle, translate=(tx, ty), scale=1.0, shear=[0.0],
                     interpolation=TF.InterpolationMode.BILINEAR, fill=0)
    mask = TF.affine(mask, angle=angle, translate=(tx, ty), scale=1.0, shear=[0.0],
                     interpolation=TF.InterpolationMode.NEAREST,  fill=0)

    if brightness:
        img = TF.adjust_brightness(img, random.uniform(1 - brightness, 1 + brightness))
    if contrast:
        img = TF.adjust_contrast(img, random.uniform(1 - contrast, 1 + contrast))

    return img, mask


def to_tensors(img, mask, img_size):
    img  = TF.resize(img,  [img_size, img_size],
                     interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.resize(mask, [img_size, img_size],
                     interpolation=TF.InterpolationMode.NEAREST)
    img_t  = TF.normalize(TF.to_tensor(img), [0.5], [0.5])
    mask_t = (TF.to_tensor(mask) > 0.5).float()
    return img_t, mask_t


# ============================================================
# U-NET (inchangé)
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = double_conv(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, 2); self.dec4 = double_conv(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, 2);  self.dec3 = double_conv(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, 2);  self.dec2 = double_conv(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, 2);   self.dec1 = double_conv(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([e4, self.up4(b)],  dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        return torch.sigmoid(self.final(d1))


# ============================================================
# LOSS ET MÉTRIQUES  (C6 : par image, puis moyennées)
# ============================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth, self.bce = smooth, nn.BCELoss()

    def forward(self, pred, target):
        bce   = self.bce(pred, target)
        p, t  = pred.view(-1), target.view(-1)
        inter = (p * t).sum()
        dice  = 1 - (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce + dice


def dice_score(pred, target, thr=0.5):
    pb    = (pred > thr).float()
    dims  = (1, 2, 3)
    inter = (pb * target).sum(dims)
    return ((2 * inter + 1e-6) /
            (pb.sum(dims) + target.sum(dims) + 1e-6)).mean()


def iou_score(pred, target, thr=0.5):
    pb    = (pred > thr).float()
    dims  = (1, 2, 3)
    inter = (pb * target).sum(dims)
    union = pb.sum(dims) + target.sum(dims) - inter
    return ((inter + 1e-6) / (union + 1e-6)).mean()


# ============================================================
# DATASET JSRT
# ============================================================
class JSRTDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=512, augment=False, verbose=True):
        self.img_size, self.augment, self.pairs = img_size, augment, []
        for fname in sorted(os.listdir(img_dir)):
            if not fname.lower().endswith('.bmp'):
                continue
            mask_path = os.path.join(mask_dir, fname.replace('.bmp', '_label.png'))
            if os.path.exists(mask_path):
                self.pairs.append((os.path.join(img_dir, fname), mask_path))
        if verbose:
            print(f"  JSRT : {len(self.pairs)} paires image/masque"
                  f" (augment={augment})")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img_pil = Image.open(img_path).convert('L')
        mask_np = np.array(Image.open(mask_path).convert('L'))
        heart   = (mask_np == HEART_VALUE).astype(np.uint8) * 255
        mask_pil = Image.fromarray(heart)

        img_pil = TF.resize(img_pil,  [self.img_size, self.img_size],
                            interpolation=TF.InterpolationMode.BILINEAR)
        mask_pil = TF.resize(mask_pil, [self.img_size, self.img_size],
                             interpolation=TF.InterpolationMode.NEAREST)

        if self.augment:
            img_pil, mask_pil = augment_pair(
                img_pil, mask_pil, rotation=15, translate=0.05,
                brightness=0.3, contrast=0.3)

        img_t  = TF.normalize(TF.to_tensor(img_pil), [0.5], [0.5])
        mask_t = (TF.to_tensor(mask_pil) > 0.5).float()
        return img_t, mask_t, 0


# ============================================================
# DATASET VINDR  (C3 : vote majoritaire)
# ============================================================
class VinDrCardioDataset(Dataset):
    def __init__(self, csv_path, img_dir, target_classes,
                 img_size=512, augment=False, min_votes=MIN_VOTES):
        self.img_dir, self.img_size = img_dir, img_size
        self.augment, self.min_votes = augment, min_votes

        df = pd.read_csv(csv_path)
        df_t = df[df['class_name'].isin(target_classes) & df['x_min'].notnull()].copy()
        self.image_ids, self.df = df_t['image_id'].unique(), df_t
        print(f"  VinDr: {len(self.image_ids)} images Cardiomegaly"
              f" (consensus ≥ {min_votes} annotateurs)")

    def __len__(self):
        return len(self.image_ids)

    def _lire_image(self, img_id):
        png_path = os.path.join('/kaggle/working/vindr_png', f"{img_id}.png")
        if os.path.exists(png_path):
            arr = np.array(Image.open(png_path).convert('L'))
            return arr, arr.shape
        try:
            ds  = pydicom.dcmread(os.path.join(self.img_dir, f"{img_id}.dicom"))
            arr = ds.pixel_array.astype(np.float32)
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255
            if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
                arr = 255 - arr
            return arr.astype(np.uint8), arr.shape
        except Exception:
            return np.zeros((512, 512), np.uint8), (512, 512)

    def _construire_masque(self, img_id, h, w):
        """
        Vote pixel par pixel entre les boîtes des annotateurs.
        Un pixel appartient au cœur s'il est couvert par au moins
        min_votes boîtes — ou par toutes, si l'image n'en compte
        qu'une ou deux.
        """
        boxes = self.df[self.df['image_id'] == img_id][
            ['x_min', 'y_min', 'x_max', 'y_max']].values
        if len(boxes) == 0:
            return np.zeros((h, w), np.uint8)

        votes = np.zeros((h, w), np.uint16)
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            x1, y1 = max(0, min(x1, w - 1)), max(0, min(y1, h - 1))
            x2, y2 = max(0, min(x2, w)),     max(0, min(y2, h))
            if x2 > x1 and y2 > y1:
                b = np.zeros((h, w), np.uint8)
                cv2.rectangle(b, (x1, y1), (x2, y2), 1, -1)
                votes += b

        seuil = min(self.min_votes, len(boxes))
        return ((votes >= seuil).astype(np.uint8) * 255)

    def __getitem__(self, idx):
        img_id      = self.image_ids[idx]
        arr, (h, w) = self._lire_image(img_id)
        mask        = self._construire_masque(img_id, h, w)

        img_pil  = Image.fromarray(arr).convert('L')
        mask_pil = Image.fromarray(mask).convert('L')

        img_pil = TF.resize(img_pil,  [self.img_size, self.img_size],
                            interpolation=TF.InterpolationMode.BILINEAR)
        mask_pil = TF.resize(mask_pil, [self.img_size, self.img_size],
                             interpolation=TF.InterpolationMode.NEAREST)

        if self.augment:
            img_pil, mask_pil = augment_pair(
                img_pil, mask_pil, rotation=10, translate=0.03,
                brightness=0.2, contrast=0.2)

        img_t  = TF.normalize(TF.to_tensor(img_pil), [0.5], [0.5])
        mask_t = (TF.to_tensor(mask_pil) > 0.5).float()
        return img_t, mask_t, 1


# ============================================================
# ENTRAÎNEMENT
# ============================================================
if __name__ == "__main__":

    print("=== CHARGEMENT DES DATASETS ===")

    # --- JSRT : deux instances sur le même répertoire, l'une augmentée ---
    jsrt_aug   = JSRTDataset(JSRT_TRAIN_IMGS, JSRT_TRAIN_MASKS, IMG_SIZE, augment=True)
    jsrt_plain = JSRTDataset(JSRT_TRAIN_IMGS, JSRT_TRAIN_MASKS, IMG_SIZE,
                             augment=False, verbose=False)

    idx = np.arange(len(jsrt_aug))
    rng = np.random.RandomState(SEED); rng.shuffle(idx)
    idx_val, idx_train = idx[:JSRT_VAL_N], idx[JSRT_VAL_N:]

    jsrt_train = Subset(jsrt_aug,   idx_train.tolist())
    jsrt_val   = Subset(jsrt_plain, idx_val.tolist())      # C4 : sans augmentation

    # --- JSRT test : strictement réservé à l'évaluation finale ---
    jsrt_test = JSRTDataset(JSRT_TEST_IMGS, JSRT_TEST_MASKS, IMG_SIZE, augment=False)

    # --- VinDr ---
    vindr_aug   = VinDrCardioDataset(VINDR_CSV, VINDR_DIR, TARGET_CLASSES,
                                     IMG_SIZE, augment=True)
    vindr_plain = VinDrCardioDataset(VINDR_CSV, VINDR_DIR, TARGET_CLASSES,
                                     IMG_SIZE, augment=False)

    n = len(vindr_aug)
    vidx = np.arange(n); rng2 = np.random.RandomState(SEED); rng2.shuffle(vidx)
    n_tr = int(0.80 * n)
    vindr_train = Subset(vindr_aug,   vidx[:n_tr].tolist())
    vindr_val   = Subset(vindr_plain, vidx[n_tr:].tolist())

    train_combined = ConcatDataset([jsrt_train, vindr_train])
    val_combined   = ConcatDataset([jsrt_val,   vindr_val])

    print(f"\nDataset combiné :")
    print(f"  JSRT train   : {len(jsrt_train)} images (masques pixel parfait)")
    print(f"  VinDr train  : {len(vindr_train)} images (consensus ≥ {MIN_VOTES})")
    print(f"  Total train  : {len(train_combined)} images")
    print(f"  JSRT val     : {len(jsrt_val)} images")
    print(f"  VinDr val    : {len(vindr_val)} images")
    print(f"  Total val    : {len(val_combined)} images")
    print(f"  Test JSRT    : {len(jsrt_test)} images  [jamais vu en validation]")

    train_loader = DataLoader(train_combined, batch_size=BATCH, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_combined,   batch_size=BATCH, shuffle=False, num_workers=2)
    test_loader  = DataLoader(jsrt_test,      batch_size=BATCH, shuffle=False, num_workers=2)

    model     = UNet().to(device)
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=PATIENCE, factor=0.5)

    best_val_loss, best_epoch, historique = float('inf'), -1, []
    CKPT_BEST  = '/kaggle/working/unet_cardio_robust_v2_best.pth'
    CKPT_FINAL = '/kaggle/working/unet_cardio_robust_v2_final.pth'   # C5

    print(f"\n{'Époque':<8} {'Train Loss':<14} {'Val Loss':<12} {'IoU':<10} {'Dice':<10} {'LR'}")
    print("-" * 72)

    for epoch in range(EPOCHS):
        model.train()
        t_loss = t_iou = t_dice = 0.0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out  = model(imgs)
            loss = criterion(out, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            t_loss += loss.item()
            t_iou  += iou_score(out.detach(), masks).item()
            t_dice += dice_score(out.detach(), masks).item()

        model.eval()
        v_loss = v_iou = v_dice = 0.0
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                out     = model(imgs)
                v_loss += criterion(out, masks).item()
                v_iou  += iou_score(out, masks).item()
                v_dice += dice_score(out, masks).item()

        tl, vl = t_loss / len(train_loader), v_loss / len(val_loader)
        vi, vd = v_iou  / len(val_loader),   v_dice / len(val_loader)
        scheduler.step(vl)
        lr_now = optimizer.param_groups[0]['lr']

        historique.append({'epoch': epoch + 1, 'train_loss': round(tl, 4),
                           'val_loss': round(vl, 4), 'val_iou': round(vi, 4),
                           'val_dice': round(vd, 4), 'lr': lr_now})

        marker = ""
        if vl < best_val_loss:
            best_val_loss, best_epoch = vl, epoch + 1
            torch.save(model.state_dict(), CKPT_BEST)
            marker = "  ← meilleur"
        print(f"{epoch+1:<8} {tl:<14.4f} {vl:<12.4f} {vi:<10.3f} {vd:<10.3f} {lr_now:.2e}{marker}")

    torch.save(model.state_dict(), CKPT_FINAL)      # C5 : fichier distinct
    with open('/kaggle/working/historique_unet_robust_v2.json', 'w') as f:
        json.dump(historique, f, indent=2)

    print(f"\nMeilleure Val Loss : {best_val_loss:.4f} (époque {best_epoch})")

    # --- Évaluation du checkpoint RETENU sur le test JSRT ---
    print("\n=== ÉVALUATION SUR TEST JSRT (checkpoint best) ===")
    model.load_state_dict(torch.load(CKPT_BEST))
    model.eval()
    test_iou = test_dice = 0.0
    with torch.no_grad():
        for imgs, masks, _ in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            test_iou  += iou_score(out, masks).item()
            test_dice += dice_score(out, masks).item()
    ti, td = test_iou / len(test_loader), test_dice / len(test_loader)
    print(f"Test IoU  : {ti:.4f}")
    print(f"Test Dice : {td:.4f}")

    # --- Courbes ---
    eps  = [x['epoch'] for x in historique]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(eps, [x['train_loss'] for x in historique], 'b-o', ms=3, label='Train Loss')
    axes[0].plot(eps, [x['val_loss']   for x in historique], 'r-o', ms=3, label='Val Loss')
    axes[0].axvline(best_epoch, color='gray', ls='--', lw=1, label=f'best (ép. {best_epoch})')
    axes[0].set_title('Loss — U-Net Robuste v2', fontweight='bold')
    axes[0].set_xlabel('Époques'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(eps, [x['val_dice'] for x in historique], 'g-o', ms=3, label='Val Dice')
    axes[1].axhline(td, color='purple', ls='--', lw=1.5, label=f'Test JSRT = {td:.4f}')
    axes[1].set_title('Dice Score — U-Net Robuste v2', fontweight='bold')
    axes[1].set_xlabel('Époques'); axes[1].set_ylim(0, 1)
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'U-Net Robuste v2 — JSRT({len(jsrt_train)}) + VinDr({len(vindr_train)})',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/courbes_unet_robust_v2.png', dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.show()

    print("\nFichiers générés :")
    print(f"  {CKPT_BEST}   ← à utiliser dans le pipeline")
    print(f"  {CKPT_FINAL}")
    print("  courbes_unet_robust_v2.png")
    print("  historique_unet_robust_v2.json")